# 30. Text Similarity

## Overview
Text similarity estimates how closely two pieces of text resemble one another. Similarity can be measured using lexical overlap, vector-space representations, or semantic embeddings.

This notebook compares TF-IDF cosine similarity with Jaccard similarity and discusses when each approach is useful.

## 1. Import libraries

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Prepare sample documents

In [2]:
documents = [
    "Machine learning helps computers learn patterns from data.",
    "Machine learning models discover patterns in data.",
    "The weather is sunny and warm today.",
    "Natural language processing analyzes human language.",
    "NLP techniques are used to analyze text and language.",
]

for index, document in enumerate(documents, start=1):
    print(f"Document {index}: {document}")

Document 1: Machine learning helps computers learn patterns from data.
Document 2: Machine learning models discover patterns in data.
Document 3: The weather is sunny and warm today.
Document 4: Natural language processing analyzes human language.
Document 5: NLP techniques are used to analyze text and language.


## 3. Calculate TF-IDF cosine similarity

Cosine similarity compares the angle between vector representations. With non-negative TF-IDF features, scores typically range from 0 to 1.

In [3]:
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(documents)

similarity_matrix = cosine_similarity(X)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=[f"doc_{i}" for i in range(1, len(documents) + 1)],
    columns=[f"doc_{i}" for i in range(1, len(documents) + 1)],
)

display(similarity_df.round(3))

,doc_1,doc_2,doc_3,doc_4,doc_5
doc_1,1.000,0.513,0.0,0.000,0.000
doc_2,0.513,1.000,0.0,0.000,0.000
doc_3,0.000,0.000,1.0,0.000,0.000
doc_4,0.000,0.000,0.0,1.000,0.213
doc_5,0.000,0.000,0.0,0.213,1.000


## 4. Find the most similar document pairs

Ignore self-similarity on the diagonal and list each pair only once.

In [4]:
pairs = []

for i in range(len(documents)):
    for j in range(i + 1, len(documents)):
        pairs.append({
            "document_1": f"doc_{i + 1}",
            "document_2": f"doc_{j + 1}",
            "similarity": similarity_matrix[i, j],
            "text_1": documents[i],
            "text_2": documents[j],
        })

pairs_df = pd.DataFrame(pairs).sort_values("similarity", ascending=False)
display(pairs_df[["document_1", "document_2", "similarity"]].head(5).round(3))

,document_1,document_2,similarity
0,doc_1,doc_2,0.513
9,doc_4,doc_5,0.213
1,doc_1,doc_3,0.000
2,doc_1,doc_4,0.000
4,doc_2,doc_3,0.000


## 5. Calculate Jaccard similarity

Jaccard similarity compares the overlap between sets of tokens. It ignores term frequency and word order.

In [5]:
def jaccard_similarity(text_a, text_b):
    tokens_a = set(text_a.lower().split())
    tokens_b = set(text_b.lower().split())

    union = tokens_a | tokens_b
    if not union:
        return 0.0

    return len(tokens_a & tokens_b) / len(union)

example_a = "machine learning finds patterns in data"
example_b = "learning models find patterns in data"

print("Jaccard similarity:", round(jaccard_similarity(example_a, example_b), 3))

Jaccard similarity: 0.5


## 6. Compare similarity methods

The Jaccard example uses whitespace tokenization for simplicity. For real text, normalize and tokenize consistently, and consider punctuation removal.

In [6]:
comparison = pd.DataFrame([
    {
        "method": "TF-IDF cosine",
        "strength": "Uses weighted vocabulary features",
        "limitation": "Misses synonyms and deeper meaning",
    },
    {
        "method": "Jaccard",
        "strength": "Simple set-overlap measure",
        "limitation": "Ignores term frequency and word order",
    },
])

display(comparison)

,method,strength,limitation
0,TF-IDF cosine,Uses weighted vocabulary features,Misses synonyms and deeper meaning
1,Jaccard,Simple set-overlap measure,Ignores term frequency and word order


## 7. Retrieve documents for a query

A query can be vectorized using the fitted TF-IDF vocabulary, then ranked by cosine similarity against the document collection.

In [7]:
query = "methods for analyzing natural language text"
query_vector = vectorizer.transform([query])
query_scores = cosine_similarity(query_vector, X).ravel()

ranking = pd.DataFrame({
    "document": documents,
    "similarity": query_scores,
}).sort_values("similarity", ascending=False)

display(ranking.round(3))

,document,similarity
3,Natural language processing analyzes human lan...,0.550
4,NLP techniques are used to analyze text and la...,0.427
0,Machine learning helps computers learn pattern...,0.000
2,The weather is sunny and warm today.,0.000
1,Machine learning models discover patterns in d...,0.000


## Interpretation and limitations

- Lexical similarity can be low even when two texts express the same idea with different words.
- Cosine similarity values are meaningful only relative to the representation and corpus.
- Semantic embedding models can capture paraphrases better, but should be chosen and evaluated for the target language and domain.

## Summary

This notebook demonstrates document similarity with TF-IDF cosine similarity, Jaccard overlap, pair ranking, and query-based retrieval.